# 第 16 课｜多头因果自注意力与 Transformer Block

**让每个位置利用前文，同时严格阻止未来信息泄漏。**

你已经用 NumPy 实现过单头注意力，用 PyTorch 学过自动求导、MLP、Embedding、交叉熵、批量训练，以及位置表示和 Pre-LN 残差子层。本课把这些组件连接起来。

本课主线：

`[B,T,D] → Q/K/V 投影 → 拆成多头 → 因果注意力 → 合并多头 → 输出投影 → 两个 Pre-LN 残差子层`

最后用一个**只有两条训练样本的机制实验**，区分“知道位置”和“能够读取前文”。它不是语言能力评测，也不是下一 token 训练。

**材料：** 本 Notebook 独立可运行；同名 `.py` 也独立可运行，不需要把两个文件放在一起。建议分两次学习：第一次完成形状与注意力，第二次完成 Block、对照实验和练习。先运行一个单元、解释结果，再进入下一个；练习答案在末尾。

## 0. 环境、文件与本课边界

沿用已有 PyTorch 环境，不需要下载模型或数据，也不需要显卡。尚未安装时，在独立单元使用 `%pip install torch`；CUDA 安装按官方选择器选择 [7]，已有环境可用时不要为本课重装。

默认 CPU；已有可用 CUDA 时把下面的 `choose_device("cpu")` 改为 `choose_device("auto")`。CPU 与不同 CUDA 后端的浮点结果可能略有差异，测试使用误差容限，不要求跨设备逐位相同。

本课是**明确的教学范围**：非空、等长、没有 padding 的序列，使用 float32/float64；不实现交叉注意力、KV cache、RoPE、GQA、混合精度或长序列高效内核。注意力模块本身不添加位置；探针在进入 Block 前只添加一次绝对位置表示。

最后保存单元会在**当前工作目录**下创建 `lesson_16_outputs`，覆盖其中同名的生成文件，不覆盖课程代码。

In [1]:
from pathlib import Path
import json
import math
import tempfile
import torch
from torch import nn
from torch.nn import functional as F

def choose_device(requested: str = "cpu") -> torch.device:
    if requested not in {"cpu", "cuda", "auto"}:
        raise ValueError("device 必须是 cpu、cuda 或 auto")
    if requested == "auto":
        requested = "cuda" if torch.cuda.is_available() else "cpu"
    if requested == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA 不可用；请用 --device cpu，或检查已有 CUDA 环境")
    return torch.device(requested)


def positive_int(value: int, name: str) -> None:
    if type(value) is not int or value <= 0:
        raise ValueError(f"{name} 必须是正整数，不接受布尔值")


def validate_dropout(p: float) -> None:
    if isinstance(p, bool) or not isinstance(p, (int, float)):
        raise ValueError("dropout 必须是 [0,1) 内的有限数")
    if not math.isfinite(p) or not 0 <= p < 1:
        raise ValueError("dropout 必须是 [0,1) 内的有限数")


def validate_float_tensor(x: torch.Tensor, ndim: int, name: str) -> None:
    if not isinstance(x, torch.Tensor) or x.ndim != ndim:
        raise ValueError(f"{name} 必须是 {ndim} 维张量")
    if any(s <= 0 for s in x.shape):
        raise ValueError(f"{name} 的所有维度必须非空")
    if x.dtype not in {torch.float32, torch.float64}:
        raise ValueError(f"{name} 的教学实现只支持 float32/float64")
    if not bool(torch.isfinite(x).all()):
        raise ValueError(f"{name} 包含 NaN 或无穷值")

torch.set_num_threads(1)  # 仅为减少微型实验的线程调度开销
DEVICE = choose_device("cpu")
print("PyTorch：", torch.__version__)
print("设备：", DEVICE, "| CUDA 可用：", torch.cuda.is_available())
print("当前工作目录：", Path.cwd())

PyTorch： 2.10.0+cpu
设备： cpu | CUDA 可用： False
当前工作目录： /mnt/data/_build16/notebook_run


## 1. Q、K、V 从哪里来？

自注意力的 Q、K、V 来自同一个输入序列，但经过**三个不同的可学习投影**。按行向量写法：

\[
Q=XW_Q+b_Q,\qquad K=XW_K+b_K,\qquad V=XW_V+b_V.
\]

Q 是用来匹配的查询表示，K 是被查询的键表示，V 是被汇总的内容表示。它们不是三个不同的数据集，也通常不等于原始 X。[1]

代码采用 `nn.Linear(D,D)`。PyTorch 内部权重形状是 `[out_features,in_features]`，实现的是 `x @ weight.T + bias`；不要把数学记号的矩阵存储方式与代码混为一谈。[8]

**先预测：** `[2,3,8]` 经过三个 `Linear(8,8)`，形状会变化吗？三个投影初始结果会相同吗？

In [2]:
torch.manual_seed(160)
x_demo = torch.randn(2, 3, 8, device=DEVICE)
q_proj = nn.Linear(8, 8).to(DEVICE)
k_proj = nn.Linear(8, 8).to(DEVICE)
v_proj = nn.Linear(8, 8).to(DEVICE)
q_demo, k_demo, v_demo = q_proj(x_demo), k_proj(x_demo), v_proj(x_demo)
print("X / Q / K / V：", x_demo.shape, q_demo.shape, k_demo.shape, v_demo.shape)
print("Q 与 K 完全相同吗？", torch.equal(q_demo, k_demo))
print("三个投影的参数数：", sum(p.numel() for m in (q_proj, k_proj, v_proj) for p in m.parameters()))

X / Q / K / V： torch.Size([2, 3, 8]) torch.Size([2, 3, 8]) torch.Size([2, 3, 8]) torch.Size([2, 3, 8])
Q 与 K 完全相同吗？ False
三个投影的参数数： 216


## 2. 多头：拆的是投影后的特征，不是把句子分成几段

约定：`B` 是批大小，`T` 是序列长度，`D` 是隐藏维度，`H` 是头数，`d=D/H` 是每个头的维度。要求 `D % H == 0`。

\[
[B,T,D]\rightarrow[B,T,H,d]\rightarrow[B,H,T,d]
\]

每个头仍然处理**全部 T 个位置**，但使用不同的投影特征。在标准 dense 投影中，每个头的特征都可以依赖输入向量的全部 D 个分量；不是只读取原始输入中一小段固定特征。[1]

下面用连续整数代替随机数，便于肉眼核对。先比较 `head 0` 与原数组每一行的前四个数，再看 `head 1`。

In [3]:
def split_heads(x: torch.Tensor, n_heads: int) -> torch.Tensor:
    """[B,T,D] -> [B,T,H,d] -> [B,H,T,d]；D=H*d。"""
    validate_float_tensor(x, 3, "x")
    positive_int(n_heads, "n_heads")
    b, t, d_model = x.shape
    if d_model % n_heads != 0:
        raise ValueError("d_model 必须能够被 n_heads 整除")
    d_head = d_model // n_heads
    return x.reshape(b, t, n_heads, d_head).transpose(1, 2)


def merge_heads(x: torch.Tensor) -> torch.Tensor:
    """[B,H,T,d] -> [B,T,H,d] -> [B,T,H*d]；是拼接，不是平均。"""
    validate_float_tensor(x, 4, "heads")
    b, h, t, d = x.shape
    return x.transpose(1, 2).contiguous().reshape(b, t, h * d)

numbered = torch.arange(24, dtype=torch.float32, device=DEVICE).reshape(1, 3, 8)
numbered_heads = split_heads(numbered, 2)
print("输入：\n", numbered)
print("拆头形状：", numbered_heads.shape)
print("head 0：\n", numbered_heads[0, 0])
print("head 1：\n", numbered_heads[0, 1])
print("合并后等于原输入：", torch.equal(merge_heads(numbered_heads), numbered))

输入：
 tensor([[[ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11., 12., 13., 14., 15.],
         [16., 17., 18., 19., 20., 21., 22., 23.]]])
拆头形状： torch.Size([1, 2, 3, 4])
head 0：
 tensor([[ 0.,  1.,  2.,  3.],
        [ 8.,  9., 10., 11.],
        [16., 17., 18., 19.]])
head 1：
 tensor([[ 4.,  5.,  6.,  7.],
        [12., 13., 14., 15.],
        [20., 21., 22., 23.]])
合并后等于原输入： True


### 2.1 `reshape` 不等于交换轴

`reshape(1,2,3,4)` 与先 `reshape(1,3,2,4)` 再 `transpose(1,2)`，**结果形状相同但元素含义不同**。

合并多头必须先从 `[B,H,T,d]` 转回 `[B,T,H,d]`，再拼成 `[B,T,D]`。不能直接按原排列展平，也不是对 H 求平均。

轴交换后，张量的内存布局可能不连续。`.contiguous()` 在需要时创建连续布局；`.reshape()` 可根据布局选择视图或复制，`.view()` 对步长有额外要求。[5] 本课用显式的 `transpose → contiguous → reshape` 帮你看清语义。不要只记“shape 对了就对了”。

In [4]:
wrong_heads = numbered.reshape(1, 2, 3, 4)
print("两个形状：", wrong_heads.shape, numbered_heads.shape)
print("内容是否相同：", torch.equal(wrong_heads, numbered_heads))
print("错误 head 0：\n", wrong_heads[0, 0])
print("正确 head 0：\n", numbered_heads[0, 0])
print("拆头结果是否连续：", numbered_heads.is_contiguous())

两个形状： torch.Size([1, 2, 3, 4]) torch.Size([1, 2, 3, 4])
内容是否相同： False
错误 head 0：
 tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.]])
正确 head 0：
 tensor([[ 0.,  1.,  2.,  3.],
        [ 8.,  9., 10., 11.],
        [16., 17., 18., 19.]])
拆头结果是否连续： False


## 3. 因果 mask：当前位置可以看自己，但不能看未来

每个头的分数形状为 `[B,H,T,T]`。倒数第二维是 query 位置，最后一维是 key 位置。

\[
S_{b,h,i,j}=\frac{q_{b,h,i}\cdot k_{b,h,j}}{\sqrt d}.
\]

在本课中，**`True` 表示允许读取**。允许条件是 `j <= i`，因此是包含对角线的下三角。先将不允许的位置填成负无穷，再对**最后一维**做 softmax。[1]

为什么能看自己？下一 token 任务中，位置 i 的输入是已知的当前 token，标签是下一 token。因果遮挡和第 13、14 课的标签移位必须同时正确；不能因为允许看自己，就把训练目标也写成当前 token。

本课没有 padding，T 大于 0，并保留对角线，每一行至少有一个允许位置。将一整行都设为负无穷会使普通 softmax 产生 NaN，后续组合 mask 时必须另行处理。

In [5]:
def causal_allowed_mask(length: int, device: torch.device | str = "cpu") -> torch.Tensor:
    """[T,T] 下三角含对角线；True=允许，False=屏蔽。"""
    positive_int(length, "length")
    return torch.ones(length, length, dtype=torch.bool, device=device).tril()

allowed_demo = causal_allowed_mask(4, DEVICE)
print("True=允许：\n", allowed_demo)
print("每行允许的 key 数：", allowed_demo.sum(dim=-1))

True=允许：
 tensor([[ True, False, False, False],
        [ True,  True, False, False],
        [ True,  True,  True, False],
        [ True,  True,  True,  True]])
每行允许的 key 数： tensor([1, 2, 3, 4])


## 4. 每个头独立做注意力

\[
A_h=\operatorname{softmax}(S_h+M),\qquad O_h=A_hV_h.
\]

关键形状：

| 对象 | 形状 |
|---|---|
| Q / K / V | `[B,H,T,d]` |
| 分数与权重 | `[B,H,T,T]` |
| 每个头的输出 | `[B,H,T,d]` |

缩放分母用 `sqrt(d)`，不是 `sqrt(D)`。下面的 `causal_attention()` 在数学部分与第 10 课一致，但现在同时处理 batch 和 head，并保留 PyTorch 的求导图。

先忽略可选 Dropout 参数，使用默认值 0。返回的第二项始终是 **Dropout 前**的权重，后面会说明原因。输入检查服务于教学，不是面向高吞吐部署的实现。

In [6]:
def causal_attention(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    dropout_p: float = 0.0,
    training: bool = False,
) -> tuple[torch.Tensor, torch.Tensor]:
    """输入均为 [B,H,T,d]；返回头输出、Dropout 前的注意力权重。

    本课限制 Q/K/V 同形状；不扩展到不等长查询、缓存或 padding。
    对角线始终允许，T>0，因此不存在一整行全被屏蔽的情况。
    """
    for name, value in (("q", q), ("k", k), ("v", v)):
        validate_float_tensor(value, 4, name)
    if q.shape != k.shape or q.shape != v.shape:
        raise ValueError("本课 Q/K/V 必须同为 [B,H,T,d] 且形状相同")
    if q.dtype != k.dtype or q.dtype != v.dtype or q.device != k.device or q.device != v.device:
        raise ValueError("Q/K/V 的 dtype 和 device 必须相同")
    validate_dropout(dropout_p)
    if type(training) is not bool:
        raise ValueError("training 必须是 bool")

    d_head = q.shape[-1]
    scores = q @ k.transpose(-2, -1) / math.sqrt(d_head)  # [B,H,T,T]
    if not bool(torch.isfinite(scores).all()):
        raise ValueError("点积分数溢出；请检查输入尺度。本课不使用低精度数值技巧")
    allowed = causal_allowed_mask(q.shape[-2], q.device)  # 广播到 [B,H,T,T]
    scores = scores.masked_fill(~allowed, float("-inf"))
    weights = torch.softmax(scores, dim=-1)              # 对 key 位置归一化
    used_weights = F.dropout(weights, p=dropout_p, training=training)
    output = used_weights @ v
    return output, weights

q_zero = torch.zeros(1, 2, 3, 2, dtype=torch.float64, device=DEVICE)
k_zero = torch.zeros_like(q_zero)
v_known = torch.tensor([[[[2., 0.], [4., 2.], [6., 4.]]]],
                       dtype=torch.float64, device=DEVICE).repeat(1, 2, 1, 1)
head_output, head_weights = causal_attention(q_zero, k_zero, v_known)
print("一个头的权重：\n", head_weights[0, 0])
print("一个头的输出：\n", head_output[0, 0])
print("权重的行和：", head_weights.sum(dim=-1))

一个头的权重：
 tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]], dtype=torch.float64)
一个头的输出：
 tensor([[2., 0.],
        [3., 1.],
        [4., 2.]], dtype=torch.float64)
权重的行和： tensor([[[1., 1., 1.],
         [1., 1., 1.]]], dtype=torch.float64)


### 4.1 手算上面的结果

Q 和 K 全为零，所以所有允许位置的分数相等。第 0 行只看 `[2,0]`；第 1 行平均 `[2,0]` 和 `[4,2]`；第 2 行平均三个向量。

预期输出是：

```text
[[2,0],
 [3,1],
 [4,2]]
```

这里两个头刻意使用相同输入，结果相同。**多头本身不保证各头自动形成不同功能**；它只提供不同投影和不同注意力分布的计算结构。请不要把每个头预先命名成“语法头”“逻辑头”等未经实验验证的角色。

## 5. 合并多头，再做输出投影

\[
\operatorname{MHA}(X)=\operatorname{Concat}(O_1,\ldots,O_H)W_O+b_O.
\]

输出投影允许不同头的特征重新组合。这里有四个 `Linear(D,D)`：Q、K、V 和输出投影。[1]

当每个 Linear 都有 bias 时，参数量是 `4*(D*D+D)`。固定 D，仅改变合法头数 H，不会改变本实现这四个投影的参数总量；但每个头的维度和 softmax 运算分组改变了，不能据此断言所有头数的计算效果相同。

本模块不做位置编码、不做 LayerNorm、不做残差。职责先分清，再组装。

In [7]:
class CausalMultiHeadSelfAttention(nn.Module):
    """四个 Linear：Q/K/V 投影 + 拼接后的输出投影。没有位置编码或残差。"""

    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0) -> None:
        super().__init__()
        positive_int(d_model, "d_model")
        positive_int(n_heads, "n_heads")
        if d_model % n_heads:
            raise ValueError("d_model 必须能被 n_heads 整除")
        validate_dropout(dropout)
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.dropout = float(dropout)
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        validate_float_tensor(x, 3, "x")
        if x.shape[-1] != self.d_model:
            raise ValueError(f"输入最后一维必须为 {self.d_model}")
        if x.device != self.q_proj.weight.device or x.dtype != self.q_proj.weight.dtype:
            raise ValueError("输入与模块的 dtype/device 必须一致")
        q = split_heads(self.q_proj(x), self.n_heads)
        k = split_heads(self.k_proj(x), self.n_heads)
        v = split_heads(self.v_proj(x), self.n_heads)
        heads, weights = causal_attention(q, k, v, self.dropout, self.training)
        output = self.out_proj(merge_heads(heads))
        return output, weights

torch.manual_seed(161)
attention_demo = CausalMultiHeadSelfAttention(8, 2).to(DEVICE).eval()
attention_output, attention_weights = attention_demo(x_demo)
print("输入：", tuple(x_demo.shape))
print("各头Q：", tuple(split_heads(attention_demo.q_proj(x_demo), 2).shape))
print("权重：", tuple(attention_weights.shape))
print("输出：", tuple(attention_output.shape))
for heads_count in (1, 2, 4):
    module = CausalMultiHeadSelfAttention(8, heads_count)
    print("H=", heads_count, "每头d=", 8 // heads_count,
          "四投影参数数=", sum(p.numel() for p in module.parameters()))

输入： (2, 3, 8)
各头Q： (2, 2, 3, 4)
权重： (2, 2, 3, 3)
输出： (2, 3, 8)
H= 1 每头d= 8 四投影参数数= 288
H= 2 每头d= 4 四投影参数数= 288
H= 4 每头d= 2 四投影参数数= 288


## 6. 与官方实现核对：不仅比较输出，也比较梯度

PyTorch 的 `F.scaled_dot_product_attention()` 接收已经投影、拆头后的 Q/K/V，不会替你创建四个 Linear。[3]

本课使用相同的输入、相同的上游梯度，核对手写版和官方版的输出及 Q/K/V 梯度。Dropout 设为 0；允许浮点误差，不要求不同后端逐位一致。

**mask 语义容易踩坑：**

| 接口 | 布尔值 True 的含义 |
|---|---|
| 本课 `causal_allowed_mask` | 允许 |
| `F.scaled_dot_product_attention` 的 `attn_mask` | 允许 |
| `nn.MultiheadAttention` 的 `attn_mask` | 屏蔽 |

不能把同一张布尔 mask 不经核对地复制到不同接口。[3][4] 后面的固定检查还会复制投影权重，与完整 `nn.MultiheadAttention` 核对。

In [8]:
torch.manual_seed(162)
original_qkv = [torch.randn(1, 2, 4, 3, device=DEVICE, dtype=torch.float64) for _ in range(3)]
manual_inputs = [x.clone().requires_grad_() for x in original_qkv]
official_inputs = [x.clone().requires_grad_() for x in original_qkv]
manual_output, _ = causal_attention(*manual_inputs)
official_output = F.scaled_dot_product_attention(*official_inputs, is_causal=True, dropout_p=0.0)
upstream = torch.randn_like(manual_output)
manual_grads = torch.autograd.grad((manual_output * upstream).sum(), manual_inputs)
official_grads = torch.autograd.grad((official_output * upstream).sum(), official_inputs)
torch.testing.assert_close(manual_output, official_output)
print("前向最大差异：", (manual_output - official_output).abs().max().item())
for name, ga, gb in zip(("Q", "K", "V"), manual_grads, official_grads):
    torch.testing.assert_close(ga, gb)
    print(name, "梯度最大差异：", (ga - gb).abs().max().item())

前向最大差异： 2.220446049250313e-16
Q 梯度最大差异： 1.6653345369377348e-16
K 梯度最大差异： 2.220446049250313e-16
V 梯度最大差异： 4.440892098500626e-16


## 7. Dropout：训练模式与评估模式要区分

`nn.Dropout` 在训练中随机置零并缩放保留项，在 `eval()` 下是恒等操作。[6] 本课注意力使用 `F.dropout(..., training=self.training)`，让模块模式控制它。

注意两点：

- Dropout **前**的注意力权重每行和为 1；Dropout 后实际参与 `@ V` 的权重，某一次采样的行和不保证为 1。本课返回前者，便于检查分布。
- `no_grad()` 不等于 `eval()`；关闭梯度不会自动关闭 Dropout。官方 SDPA 也是根据显式传入的 `dropout_p` 决定是否启用，调用者在评估时应传 0。[3]

后面的训练探针使用 `dropout=0`，以排除随机遮挡对机制对照的影响。

In [9]:
drop_demo = CausalMultiHeadSelfAttention(8, 2, dropout=0.5).to(DEVICE)
drop_demo.load_state_dict(attention_demo.state_dict())
drop_demo.eval()
with torch.no_grad():
    eval_a, eval_weights = drop_demo(x_demo)
    eval_b, _ = drop_demo(x_demo)
print("eval 两次输出一致：", torch.equal(eval_a, eval_b))
drop_demo.train()
with torch.no_grad():
    train_a, pre_drop_weights = drop_demo(x_demo)
    train_b, _ = drop_demo(x_demo)
print("no_grad 但 train 模式，输出仍可不同：", not torch.equal(train_a, train_b))
print("返回的权重行和：", pre_drop_weights.sum(-1))
once_used_weights = F.dropout(pre_drop_weights, p=0.5, training=True)
print("另一次独立 Dropout 后的行和：", once_used_weights.sum(-1))

eval 两次输出一致： True
no_grad 但 train 模式，输出仍可不同： True
返回的权重行和： tensor([[[1., 1., 1.],
         [1., 1., 1.]],

        [[1., 1., 1.],
         [1., 1., 1.]]])
另一次独立 Dropout 后的行和： tensor([[[2.0000, 0.9361, 0.0000],
         [2.0000, 1.6201, 0.9013]],

        [[2.0000, 1.3700, 1.4399],
         [2.0000, 0.7684, 0.5047]]])


## 8. 组装一个 decoder-only Pre-LN Transformer Block

忽略 Dropout、只看主干公式：

\[
U=X+\operatorname{MHA}(\operatorname{LN}_1(X)),
\]
\[
Y=U+\operatorname{FFN}(\operatorname{LN}_2(U)).
\]

这是**两个残差子层**，两个独立 LayerNorm；第二个子层处理的是更新后的 U，而不是旧的 X。FFN 使用上一课的 `Linear → GELU → Linear`，只在同一位置内变换特征。

原始 Transformer 使用 Post-LN；本课程选择 Pre-LN，属于不同的归一化组织方式，不宣称精确复现原始整套架构。[1][2]

这里组装的是 decoder-only 块，没有 encoder-decoder 交叉注意力。Block 的输入和输出都是隐藏状态 `[B,T,D]`，**不是 token ID 或词表概率**；完整语言模型还要有输入 Embedding、位置表示、多层堆叠、最终归一化及词表输出头。

In [10]:
class TransformerBlock(nn.Module):
    def __init__(
        self, d_model: int, n_heads: int, hidden_dim: int, dropout: float = 0.0
    ) -> None:
        super().__init__()
        positive_int(hidden_dim, "hidden_dim")
        self.attn = CausalMultiHeadSelfAttention(d_model, n_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.fc1 = nn.Linear(d_model, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, d_model)
        self.residual_dropout = nn.Dropout(dropout)

    def forward(
        self, x: torch.Tensor, return_weights: bool = False
    ) -> torch.Tensor | tuple[torch.Tensor, torch.Tensor]:
        # 第一个子层：每个位置可以读取自己和前文。
        attn_output, weights = self.attn(self.norm1(x))
        x = x + self.residual_dropout(attn_output)
        # 第二个子层：逐位置变换已汇总的特征，保留新的 x 作为残差。
        ffn_output = self.fc2(F.gelu(self.fc1(self.norm2(x))))
        x = x + self.residual_dropout(ffn_output)
        if return_weights:
            return x, weights
        return x

torch.manual_seed(163)
block_demo = TransformerBlock(8, 2, 16).to(DEVICE).eval()
block_output, block_weights = block_demo(x_demo, return_weights=True)
print("块输入 / 输出：", x_demo.shape, block_output.shape)
print("块返回的注意力权重：", block_weights.shape)
print("块的参数数：", sum(p.numel() for p in block_demo.parameters()))

块输入 / 输出： torch.Size([2, 3, 8]) torch.Size([2, 3, 8])
块返回的注意力权重： torch.Size([2, 2, 3, 3])
块的参数数： 600


## 9. 测试一：改未来，过去必须不变

在 `eval()`、无随机 Dropout 的条件下，保持位置 0、1 不变，只修改位置 2 及以后。比较整个 Block 在前两个位置的输出，而不是只看 mask 的图案。

还要比较：完整序列的前缀输出，是否等于单独输入该前缀的输出。本课每个 token 的 LayerNorm 只处理特征维，FFN 也逐位置，因此不会绕过注意力 mask 引入未来信息。

这是更强的检查思路：**检查整个数据路径，而不只检查某一张 mask 是否正确。**

In [11]:
torch.manual_seed(164)
causal_x = torch.randn(2, 4, 8, device=DEVICE)
changed_future_x = causal_x.clone()
changed_future_x[:, 2:] += torch.randn_like(changed_future_x[:, 2:]) * 5
with torch.no_grad():
    full_out = block_demo(causal_x)
    changed_out = block_demo(changed_future_x)
    prefix_out = block_demo(causal_x[:, :2])
print("改未来后前缀最大差异：", (full_out[:, :2] - changed_out[:, :2]).abs().max().item())
print("单独前缀与完整前缀最大差异：", (full_out[:, :2] - prefix_out).abs().max().item())
torch.testing.assert_close(full_out[:, :2], changed_out[:, :2])
torch.testing.assert_close(full_out[:, :2], prefix_out)

x_for_grad = causal_x.clone().requires_grad_()
block_demo(x_for_grad)[:, :2].square().sum().backward()
print("前缀损失对未来输入的最大梯度：", x_for_grad.grad[:, 2:].abs().max().item())

改未来后前缀最大差异： 0.0
单独前缀与完整前缀最大差异： 2.384185791015625e-07
前缀损失对未来输入的最大梯度： 0.0


## 10. 测试二：改过去，后面确实可以受影响

“不看未来”和“能看过去”是两件事。一个完全不做注意力的逐位置模型，也可以通过“不泄漏未来”的检查。

先做一个可手算的计算实验：仍用 Q=K=0，令 V 的第 0 个位置增加 `[6,6]`。最后位置对三个允许位置等权平均，所以最后输出应增加 `[2,2]`。之后再训练模型，让它学会用这个跨位置通道完成任务。

In [12]:
changed_past_v = v_known.clone()
changed_past_v[:, :, 0] += 6
new_head_output, _ = causal_attention(q_zero, k_zero, changed_past_v)
print("最后位置输出的变化：", new_head_output[:, :, -1] - head_output[:, :, -1])
torch.testing.assert_close(new_head_output[:, :, -1] - head_output[:, :, -1],
                           torch.full_like(head_output[:, :, -1], 2.0))

最后位置输出的变化： tensor([[[2., 2.],
         [2., 2.]]], dtype=torch.float64)


## 11. 验证两条残差主路和可训练参数

把注意力的输出投影、FFN 的第二层都置零（包含 bias），两个分支就都输出零，此时整个 Block 应等于输入，且 `output.sum()` 对输入的梯度应全为 1。

这只是测试副本，不是在建议把训练模型全部归零初始化。全零分支会让本轮某些更早的分支参数梯度为零。

再用正常初始化检查 Q/K/V、输出投影、FFN 的权重梯度。**“可训练参数”不意味着每个参数在任何输入上都有非零梯度。** 例如部分注意力偏置在特定结构下可能因 softmax 的平移不变性而没有有效梯度，因此这里检查选定权重，不要求全部 bias 都非零。

In [13]:
zero_block = TransformerBlock(8, 2, 16).to(DEVICE)
with torch.no_grad():
    zero_block.attn.out_proj.weight.zero_()
    zero_block.attn.out_proj.bias.zero_()
    zero_block.fc2.weight.zero_()
    zero_block.fc2.bias.zero_()
identity_input = causal_x.clone().requires_grad_()
identity_output = zero_block(identity_input)
identity_output.sum().backward()
print("分支为零时输出等于输入：", torch.equal(identity_output, identity_input))
print("直通梯度全为1：", torch.equal(identity_input.grad, torch.ones_like(identity_input)))

block_demo.zero_grad(set_to_none=True)
weighted_loss = (block_demo(causal_x) * torch.randn_like(causal_x)).sum()
weighted_loss.backward()
for name in ("attn.q_proj.weight", "attn.k_proj.weight", "attn.v_proj.weight",
             "attn.out_proj.weight", "fc1.weight", "fc2.weight"):
    parameter = dict(block_demo.named_parameters())[name]
    print(name, "梯度范数：", parameter.grad.norm().item())

分支为零时输出等于输入： True
直通梯度全为1： True
attn.q_proj.weight 梯度范数： 1.4076528549194336
attn.k_proj.weight 梯度范数： 1.2772045135498047
attn.v_proj.weight 梯度范数： 7.751506805419922
attn.out_proj.weight 梯度范数： 8.355426788330078
fc1.weight 梯度范数： 7.4688496589660645
fc2.weight 梯度范数： 11.58383846282959


## 12. 机制探针：相同末 token，预测不同前文

本课只有三个符号：`A=0`、`B=1`、`?=2`。输入与分类标签为：

| 完整输入 | 最后位置需要预测的类别 |
|---|---|
| `A ?` / `[0,2]` | 0（A） |
| `B ?` / `[1,2]` | 1（B） |

只在 `?` 的输出上计算**二分类交叉熵**。这不是把标签右移构成的语言模型训练；输出头维度是 2，不是词表大小 3。目标是观察信息能否从第 0 个位置传到第 1 个位置。

两组都有 token Embedding、位置 Embedding、逐位置 FFN、最终 LayerNorm 和分类头。注意力组额外包含注意力子层；公共部分不仅结构一致，初始权重也复制为一致。**参数量并不相等**，这个对照验证信息通道，不用来比较单位参数效率。

两条输入都用于训练，没有验证集和测试集。即使达到 100%，也只说明拟合了这两个样本，不能证明泛化、推理或语言理解能力。

In [14]:
class TokenwiseBlock(nn.Module):
    """对照组：保留同名 FFN 子层，但完全没有跨位置交互。"""

    def __init__(self, d_model: int, hidden_dim: int) -> None:
        super().__init__()
        self.norm2 = nn.LayerNorm(d_model)
        self.fc1 = nn.Linear(d_model, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.fc2(F.gelu(self.fc1(self.norm2(x))))


class ContextProbe(nn.Module):
    """词表 A=0、B=1、?=2；输出两个分类 logits，不是词表输出头。"""

    def __init__(
        self, use_attention: bool, d_model: int = 16, n_heads: int = 4,
        hidden_dim: int = 32, max_len: int = 2,
    ) -> None:
        super().__init__()
        if type(use_attention) is not bool:
            raise ValueError("use_attention 必须是 bool")
        for name, value in (("d_model", d_model), ("hidden_dim", hidden_dim),
                            ("max_len", max_len), ("n_heads", n_heads)):
            positive_int(value, name)
        if d_model % n_heads:
            raise ValueError("d_model 必须能被 n_heads 整除")
        self.config = dict(use_attention=use_attention, d_model=d_model,
                           n_heads=n_heads, hidden_dim=hidden_dim, max_len=max_len)
        self.max_len = max_len
        self.token_embedding = nn.Embedding(3, d_model)
        self.position_embedding = nn.Embedding(max_len, d_model)
        if use_attention:
            self.block = TransformerBlock(d_model, n_heads, hidden_dim, dropout=0.0)
        else:
            self.block = TokenwiseBlock(d_model, hidden_dim)
        self.final_norm = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, 2)

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        if not isinstance(ids, torch.Tensor) or ids.ndim != 2 or ids.dtype != torch.long:
            raise ValueError("ids 必须为 torch.long 类型的 [B,T]")
        if any(s <= 0 for s in ids.shape) or ids.shape[1] > self.max_len:
            raise ValueError("批次/序列不可为空，且 T 不能超过 max_len")
        if bool((ids < 0).any()) or bool((ids >= 3).any()):
            raise ValueError("探针 token ID 只能是 0、1、2")
        if ids.device != self.token_embedding.weight.device:
            raise ValueError("ids 与模型应位于同一设备")
        positions = torch.arange(ids.shape[1], device=ids.device)
        x = self.token_embedding(ids) + self.position_embedding(positions).unsqueeze(0)
        return self.classifier(self.final_norm(self.block(x)))


def make_probe_pair(seed: int, device: torch.device) -> dict[str, ContextProbe]:
    """共同组件不仅结构相同，初始参数也相同；不宣称两组参数量相等。"""
    torch.manual_seed(seed)
    full = ContextProbe(True).to(device)
    baseline = ContextProbe(False).to(device)
    source = full.state_dict()
    baseline.load_state_dict({name: source[name].clone() for name in baseline.state_dict()})
    return {"tokenwise": baseline, "attention": full}

initial_pair = make_probe_pair(1601, DEVICE)
probe_ids = torch.tensor([[0, 2], [1, 2]], dtype=torch.long, device=DEVICE)
for name, model in initial_pair.items():
    model.eval()
    with torch.no_grad():
        initial_logits = model(probe_ids)[:, -1]
    print(name, "参数数：", sum(p.numel() for p in model.parameters()),
          "两个末位置 logits 最大差异：", (initial_logits[0] - initial_logits[1]).abs().max().item())

tokenwise 参数数： 1250 两个末位置 logits 最大差异： 0.0
attention 参数数： 2370 两个末位置 logits 最大差异： 0.05344247817993164


### 12.1 为什么逐位置模型的损失不能低于 log(2)？

两个末位置都是同一个 token `?`、同一个位置编号 1。逐位置模型无论堆多宽的 FFN，在这里都只能给出同一个类别分布 `[p,1-p]`。

两个标签各出现一次，因此：

\[
L=-\frac12(\log p+\log(1-p))\ge\log 2.
\]

当 `p=0.5` 时达到下界。使用相同的 argmax 规则，只能正确一个样本，准确率为 50%。

这条限制来自**本探针的模型结构与输入约定**。加入注意力后，末位置可以获得不同的前文，限制不再成立；但是否实际学到，要看运行结果，不把收敛写成单元测试的必然条件。

In [15]:
def evaluate_probe(model: ContextProbe, ids: torch.Tensor, labels: torch.Tensor) -> dict:
    previous_mode = model.training
    model.eval()
    try:
        with torch.no_grad():
            logits = model(ids)[:, -1, :]
            return {
                "loss": float(F.cross_entropy(logits, labels).item()),
                "accuracy": float((logits.argmax(-1) == labels).float().mean().item()),
                "predictions": logits.argmax(-1).cpu().tolist(),
                "probabilities": logits.softmax(-1).cpu().tolist(),
                "last_position_logit_difference": float((logits[0] - logits[1]).abs().max().item()),
            }
    finally:
        model.train(previous_mode)


def run_context_experiment(
    device: torch.device, steps: int = 160, seed: int = 1601, lr: float = 0.02
) -> tuple[dict[str, ContextProbe], dict]:
    positive_int(steps, "steps")
    if isinstance(lr, bool) or not isinstance(lr, (int, float)) or not math.isfinite(lr) or lr <= 0:
        raise ValueError("lr 必须为有限正数")
    if type(seed) is not int or seed < 0:
        raise ValueError("seed 必须为非负整数")
    ids = torch.tensor([[0, 2], [1, 2]], dtype=torch.long, device=device)
    labels = torch.tensor([0, 1], dtype=torch.long, device=device)
    models = make_probe_pair(seed, device)
    results = {
        "scope": "只有两条训练样本、只监督末位置分类；没有验证/测试集，不是语言模型能力评测",
        "inputs": ids.cpu().tolist(), "labels": labels.cpu().tolist(),
        "id_to_token": ["A", "B", "?"],
        "settings": {"steps": steps, "seed": seed, "lr": lr, "dropout": 0.0},
        "environment": {"torch": str(torch.__version__), "device": str(device)},
        "models": {},
    }
    for name, model in models.items():
        before = evaluate_probe(model, ids, labels)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        history = []
        model.train()
        for step in range(1, steps + 1):
            optimizer.zero_grad(set_to_none=True)
            logits = model(ids)[:, -1, :]
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            optimizer.step()
            if step == 1 or step % 20 == 0 or step == steps:
                history.append({"step": step, **evaluate_probe(model, ids, labels)})
        model.eval()
        after = evaluate_probe(model, ids, labels)
        changed_future = ids.clone()
        changed_future[:, 1] = 0
        with torch.no_grad():
            before_future = model(ids)[:, 0, :]
            after_future = model(changed_future)[:, 0, :]
        results["models"][name] = {
            "config": model.config,
            "parameters": sum(p.numel() for p in model.parameters()),
            "before": before, "after": after, "history": history,
            "future_change_prefix_max_diff": float((before_future - after_future).abs().max().item()),
        }
    return models, results

models, results = run_context_experiment(DEVICE, steps=160, seed=1601, lr=0.02)
for name, metrics in results["models"].items():
    print(name)
    print("  训练前 loss：", metrics["before"]["loss"])
    print("  训练后 loss：", metrics["after"]["loss"])
    print("  训练准确率：", metrics["after"]["accuracy"])
    print("  预测类别：", metrics["after"]["predictions"])
print("逐位置模型的理论下界：", math.log(2))

tokenwise
  训练前 loss： 0.7272026538848877
  训练后 loss： 0.6931472420692444
  训练准确率： 0.5
  预测类别： [1, 1]
attention
  训练前 loss： 0.7470948100090027
  训练后 loss： 3.379520057933405e-05
  训练准确率： 1.0
  预测类别： [0, 1]
逐位置模型的理论下界： 0.6931471805599453


### 12.2 用干预检查学习结果，而不是只看准确率

比较 `A ?` 与 `B ?` 的最后输出：最后 token 与位置都不变，只改前文，注意力模型训练后应有明显的分类变化。

再比较 `A ?` 与 `A A` 的**第 0 个位置**输出：只改未来，前面的输出必须不变。注意第 0 个位置没有被分类损失直接监督，我们这里只检查计算依赖，不解读它的分类语义。

`run_context_experiment()` 已记录这些值。打印它们，试着解释两个方向的差异为什么不同。

In [16]:
for name, metrics in results["models"].items():
    print(name)
    print("  两条前文对应的末位置概率：", metrics["after"]["probabilities"])
    print("  改前文时末位置 logits 最大差异：", metrics["after"]["last_position_logit_difference"])
    print("  改未来时前缀 logits 最大差异：", metrics["future_change_prefix_max_diff"])
print("数据说明：", results["scope"])

tokenwise
  两条前文对应的末位置概率： [[0.4999975264072418, 0.5000024437904358], [0.4999975264072418, 0.5000024437904358]]
  改前文时末位置 logits 最大差异： 0.0
  改未来时前缀 logits 最大差异： 0.0
attention
  两条前文对应的末位置概率： [[0.9999771118164062, 2.2828256987850182e-05], [4.46433768956922e-05, 0.9999552965164185]]
  改前文时末位置 logits 最大差异： 10.864264488220215
  改未来时前缀 logits 最大差异： 0.0
数据说明： 只有两条训练样本、只监督末位置分类；没有验证/测试集，不是语言模型能力评测


## 13. 固定检查：覆盖形状、数值、梯度和信息流

下面是与脚本相同的检查代码。可以先运行，再按主题回读。检查包括：拆合头的元素对应关系、未来权重为零、行和、T=1、非法输入、官方 SDPA 的前向/梯度、完整 MHA 权重映射、Dropout、整个 Block 和两层堆叠的因果性、残差直通梯度、样本间隔离，以及保存重载。

训练是否恰好收敛到某个准确率，不属于这些固定测试。测试是有限的，不能证明对所有输入都正确。

检查器显式抛出 `AssertionError`，不依赖会被 Python `-O` 移除的 `assert` 语句。

In [17]:
class CheckBook:
    def __init__(self) -> None:
        self.names: list[str] = []

    def ok(self, name: str, condition: bool) -> None:
        if not bool(condition):
            raise AssertionError(name)
        self.names.append(name)

    def close(self, name: str, actual: torch.Tensor, expected: torch.Tensor,
              rtol: float = 1e-6, atol: float = 1e-7) -> None:
        try:
            torch.testing.assert_close(actual, expected, rtol=rtol, atol=atol)
        except AssertionError as error:
            raise AssertionError(name) from error
        self.names.append(name)

    def raises(self, name: str, function, exception=ValueError) -> None:
        try:
            function()
        except exception:
            self.names.append(name)
        else:
            raise AssertionError(f"{name}：应抛出 {exception.__name__}")


def run_checks(device: torch.device = torch.device("cpu")) -> list[str]:
    """每项通过后才登记；任何失败会立即停止，不依赖可被 -O 移除的 assert。"""
    torch.manual_seed(1600)
    c = CheckBook()
    x = torch.arange(48, dtype=torch.float64, device=device).reshape(2, 3, 8)
    heads = split_heads(x, 2)
    c.ok("拆头形状", heads.shape == (2, 2, 3, 4))
    c.close("头0对应每个token前4维", heads[:, 0], x[:, :, :4])
    c.close("头1对应每个token后4维", heads[:, 1], x[:, :, 4:])
    c.close("合并是拆头的逆操作", merge_heads(heads), x)
    noncontig = torch.randn(2, 8, 3, dtype=torch.float64, device=device).transpose(1, 2)
    c.close("非连续输入也可正确拆合", merge_heads(split_heads(noncontig, 2)), noncontig)
    dx = x.clone().requires_grad_()
    merge_heads(split_heads(dx, 2)).sum().backward()
    c.close("拆合不切断梯度", dx.grad, torch.ones_like(dx))
    c.raises("拒绝头数不能整除", lambda: split_heads(x, 3))
    c.raises("拒绝布尔头数", lambda: split_heads(x, True))
    c.raises("拒绝零头数", lambda: split_heads(x, 0))
    c.raises("拒绝错误维数", lambda: split_heads(x[0], 2))
    c.raises("拒绝整数隐藏状态", lambda: split_heads(x.long(), 2))
    c.raises("拒绝空序列", lambda: split_heads(x[:, :0], 2))
    c.raises("拒绝错误合并维数", lambda: merge_heads(x))
    c.ok("直接reshape不能替代轴交换", not torch.equal(x.reshape(2, 2, 3, 4), heads))

    allowed = causal_allowed_mask(3, device)
    c.close("因果掩码含对角线", allowed, torch.tensor(
        [[True, False, False], [True, True, False], [True, True, True]], device=device))
    c.ok("每行至少允许一个位置", bool(allowed.any(-1).all()))
    c.raises("拒绝零长度mask", lambda: causal_allowed_mask(0, device))
    q = torch.zeros(1, 2, 3, 2, dtype=torch.float64, device=device)
    k = q.clone()
    v = torch.tensor([[[[2., 0.], [4., 2.], [6., 4.]]]], device=device,
                     dtype=torch.float64).repeat(1, 2, 1, 1)
    out, weights = causal_attention(q, k, v)
    expected = torch.tensor([[[[2., 0.], [3., 1.], [4., 2.]]]],
                            device=device, dtype=torch.float64).repeat(1, 2, 1, 1)
    c.close("零分数等权汇总允许前缀", out, expected)
    c.close("权重逐行归一化", weights.sum(-1), torch.ones(1, 2, 3, device=device, dtype=torch.float64))
    c.ok("未来权重严格为零", bool((weights.masked_select(~allowed) == 0).all()))
    c.ok("第一位置只看自己", bool((weights[:, :, 0, 0] == 1).all()))
    c.close("T=1的头输出就是V", causal_attention(q[:, :, :1], k[:, :, :1], v[:, :, :1])[0], v[:, :, :1])
    c.raises("拒绝不等长QKV", lambda: causal_attention(q, k[:, :, :2], v))
    c.raises("拒绝混合dtype", lambda: causal_attention(q, k.float(), v))
    c.raises("拒绝NaN输入", lambda: causal_attention(q + float("nan"), k, v))
    c.raises("拒绝非法dropout", lambda: causal_attention(q, k, v, dropout_p=1.0))

    # 与官方 SDPA 对齐前向和反向，使用同一个输出上游梯度。
    original = [torch.randn(2, 2, 4, 3, device=device, dtype=torch.float64) for _ in range(3)]
    manual_inputs = [a.clone().requires_grad_() for a in original]
    official_inputs = [a.clone().requires_grad_() for a in original]
    manual, manual_w = causal_attention(*manual_inputs)
    official = F.scaled_dot_product_attention(*official_inputs, dropout_p=0.0, is_causal=True)
    c.close("与官方SDPA前向一致", manual, official)
    upstream = torch.randn_like(manual)
    grads_a = torch.autograd.grad((manual * upstream).sum(), manual_inputs)
    grads_b = torch.autograd.grad((official * upstream).sum(), official_inputs)
    for name, a, b in zip(("Q", "K", "V"), grads_a, grads_b):
        c.close(f"与官方SDPA的{name}梯度一致", a, b)
    boolean_ref = F.scaled_dot_product_attention(
        *original, attn_mask=causal_allowed_mask(4, device), dropout_p=0.0)
    c.close("SDPA布尔True表示允许", manual.detach(), boolean_ref)
    # 与标准同尺度计算核对，用不同D与d以发现sqrt(D)写错。
    raw = original[0] @ original[1].transpose(-2, -1) / math.sqrt(3)
    raw = raw.masked_fill(~causal_allowed_mask(4, device), float("-inf"))
    c.close("缩放使用每头维度", manual_w, raw.softmax(-1))
    changed_k, changed_v = original[1].clone(), original[2].clone()
    changed_k[:, :, 2:] += 3
    changed_v[:, :, 2:] -= 7
    perturbed = causal_attention(original[0], changed_k, changed_v)[0]
    c.close("未来KV不影响前缀", perturbed[:, :, :2], manual.detach()[:, :, :2])
    k_grad = original[1].clone().requires_grad_()
    v_grad = original[2].clone().requires_grad_()
    prefix = causal_attention(original[0], k_grad, v_grad)[0][:, :, :2].square().sum()
    gk, gv = torch.autograd.grad(prefix, (k_grad, v_grad))
    c.close("前缀损失对未来K梯度为0", gk[:, :, 2:], torch.zeros_like(gk[:, :, 2:]))
    c.close("前缀损失对未来V梯度为0", gv[:, :, 2:], torch.zeros_like(gv[:, :, 2:]))
    v_past = v.clone()
    v_past[:, :, 0] += 6
    c.close("改变过去能影响后位", causal_attention(q, k, v_past)[0][:, :, -1] - out[:, :, -1],
            torch.full_like(out[:, :, -1], 2.0))

    # nn.MultiheadAttention：布尔attn_mask的True反而表示屏蔽。
    attn = CausalMultiHeadSelfAttention(8, 2).to(device=device, dtype=torch.float64).eval()
    xx = torch.randn(2, 4, 8, device=device, dtype=torch.float64)
    yy, ww = attn(xx)
    c.ok("MHA输出形状", yy.shape == (2, 4, 8))
    c.ok("每头权重形状", ww.shape == (2, 2, 4, 4))
    c.ok("MHA四投影参数量", sum(p.numel() for p in attn.parameters()) == 4 * (8 * 8 + 8))
    reference = nn.MultiheadAttention(8, 2, batch_first=True, dropout=0.0).to(device=device, dtype=torch.float64).eval()
    with torch.no_grad():
        reference.in_proj_weight.copy_(torch.cat([attn.q_proj.weight, attn.k_proj.weight, attn.v_proj.weight]))
        reference.in_proj_bias.copy_(torch.cat([attn.q_proj.bias, attn.k_proj.bias, attn.v_proj.bias]))
        reference.out_proj.load_state_dict(attn.out_proj.state_dict())
    ref_y, ref_w = reference(xx, xx, xx, attn_mask=~causal_allowed_mask(4, device),
                             need_weights=True, average_attn_weights=False)
    c.close("与官方完整MHA输出一致", yy, ref_y)
    c.close("与官方完整MHA每头权重一致", ww, ref_w)
    attn.zero_grad(set_to_none=True)
    (yy * torch.randn_like(yy)).sum().backward()
    for name in ("q_proj", "k_proj", "v_proj", "out_proj"):
        grad = getattr(attn, name).weight.grad
        c.ok(f"{name}权重有非零有限梯度", grad is not None and bool(torch.isfinite(grad).all()) and float(grad.norm()) > 0)
    c.raises("模块拒绝D/H不整除", lambda: CausalMultiHeadSelfAttention(7, 2))
    c.raises("模块拒绝输入最后维不符", lambda: attn(xx[:, :, :7]))
    c.raises("模块拒绝dtype不一致", lambda: attn(xx.float()))
    c.raises("模块拒绝负dropout", lambda: CausalMultiHeadSelfAttention(8, 2, -0.1))
    c.raises("模块拒绝布尔dropout", lambda: CausalMultiHeadSelfAttention(8, 2, True))

    dropout_attn = CausalMultiHeadSelfAttention(8, 2, 0.5).to(device=device, dtype=torch.float64)
    dropout_attn.load_state_dict(attn.state_dict())
    dropout_attn.eval()
    c.close("eval关闭注意力dropout", dropout_attn(xx)[0], attn(xx)[0])
    c.close("eval重复结果相同", dropout_attn(xx)[0], dropout_attn(xx)[0])
    dropout_attn.train()
    train_y, train_w = dropout_attn(xx)
    c.ok("train输出保持有限", bool(torch.isfinite(train_y).all()))
    c.close("返回的是dropout前权重", train_w, ww)

    # 完整块、堆叠块的因果性，以及残差恒等路径。
    block = TransformerBlock(8, 2, 16).to(device=device, dtype=torch.float64).eval()
    block_out, block_w = block(xx, return_weights=True)
    c.ok("块输出保持BTD", block_out.shape == xx.shape)
    c.ok("块可返回每头权重", block_w.shape == (2, 2, 4, 4))
    c.ok("块参数量公式", sum(p.numel() for p in block.parameters()) == 4*(8*8+8)+4*8+2*8*16+16+8)
    future_x = xx.clone()
    future_x[:, 2:] += torch.randn_like(future_x[:, 2:]) * 5
    c.close("整个块不泄露未来", block(future_x)[:, :2], block_out[:, :2])
    c.close("前缀单独计算与完整前缀一致", block(xx[:, :2]), block_out[:, :2])
    stacked = nn.Sequential(block, TransformerBlock(8, 2, 16).to(device=device, dtype=torch.float64).eval())
    c.close("两层堆叠保持因果性", stacked(future_x)[:, :2], stacked(xx)[:, :2])
    independent = xx.clone()
    independent[1] += 8
    c.close("不同batch样本不混合", block(independent)[0], block_out[0])
    xg = xx.clone().requires_grad_()
    block(xg)[:, :2].square().sum().backward()
    c.close("完整块前缀对未来输入梯度为0", xg.grad[:, 2:], torch.zeros_like(xg.grad[:, 2:]))
    zero_block = TransformerBlock(8, 2, 16).to(device=device, dtype=torch.float64)
    with torch.no_grad():
        zero_block.attn.out_proj.weight.zero_()
        zero_block.attn.out_proj.bias.zero_()
        zero_block.fc2.weight.zero_()
        zero_block.fc2.bias.zero_()
    identity_x = xx.clone().requires_grad_()
    identity_y = zero_block(identity_x)
    c.close("两个分支为0时块为恒等映射", identity_y, identity_x)
    identity_y.sum().backward()
    c.close("恒等主路梯度保留", identity_x.grad, torch.ones_like(identity_x))

    pair = make_probe_pair(1601, device)
    full_state = pair["attention"].state_dict()
    c.ok("两组公共参数初值相同", all(torch.equal(value, full_state[name])
         for name, value in pair["tokenwise"].state_dict().items()))
    ids = torch.tensor([[0, 2], [1, 2]], device=device)
    base_logits = pair["tokenwise"](ids)
    c.close("逐位置模型无法区分相同末token", base_logits[0, -1], base_logits[1, -1])
    c.ok("探针输出不是词表维度", base_logits.shape == (2, 2, 2))
    for name, model in pair.items():
        model.train()
        evaluate_probe(model, ids, torch.tensor([0, 1], device=device))
        c.ok(f"{name}验证恢复训练模式", model.training)
    c.raises("探针拒绝越界token", lambda: pair["attention"](torch.tensor([[0, 3]], device=device)))
    c.raises("探针拒绝过长序列", lambda: pair["attention"](torch.tensor([[0, 1, 2]], device=device)))
    c.raises("探针拒绝非long编号", lambda: pair["attention"](ids.float()))
    with tempfile.TemporaryDirectory() as temp:
        path = Path(temp) / "model.pt"
        torch.save({k: v.detach().cpu() for k, v in block.state_dict().items()}, path)
        restored_block = TransformerBlock(8, 2, 16).double().eval()
        restored_block.load_state_dict(torch.load(path, weights_only=True, map_location="cpu"))
        c.close("保存加载块保持输出", restored_block(xx.cpu()), block_out.detach().cpu(), rtol=1e-5, atol=1e-6)
    return c.names

check_names = run_checks(DEVICE)
results["checks"] = {"count": len(check_names), "names": check_names}
print("通过固定检查：", len(check_names), "项")

通过固定检查： 73 项


### 13.1 保存、重新构造并核对输出

保存模型配置、两个模型的 `state_dict` 和实验指标；权重先移到 CPU，便于以后在其他设备重建。只加载自己生成或可信来源的检查点，这里显式使用 `weights_only=True`。

`save_experiment()` 不只检查文件存在，还会重建模型、加载刚保存的权重，并核对输出。默认覆盖 `lesson_16_outputs/context_probe.pt` 与 `metrics.json`。

也可以在终端运行同名脚本：

```bash
python lesson_16_multihead_attention_and_transformer_block.py
python lesson_16_multihead_attention_and_transformer_block.py --checks-only
python lesson_16_multihead_attention_and_transformer_block.py --device auto
```

脚本另支持 `--steps`、`--seed`、`--lr`、`--output-dir`。不是所有参数组合都会收敛；改变设置后先观察，不改测试预期来迎合结果。

In [18]:
def save_experiment(models: dict[str, ContextProbe], results: dict, output_dir: Path) -> None:
    """只保存参数/配置；重载刚保存的可信检查点并核对预测。"""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    payload = {}
    expected_logits = {}
    for name, model in models.items():
        model.eval()
        ids = torch.tensor(results["inputs"], device=next(model.parameters()).device)
        with torch.no_grad():
            expected_logits[name] = model(ids).cpu()
        payload[name] = {
            "config": model.config,
            "state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
        }
    path = output_dir / "context_probe.pt"
    torch.save(payload, path)
    restored = torch.load(path, map_location="cpu", weights_only=True)
    ids_cpu = torch.tensor(results["inputs"], dtype=torch.long)
    for name, item in restored.items():
        model = ContextProbe(**item["config"]).eval()
        model.load_state_dict(item["state_dict"])
        with torch.no_grad():
            torch.testing.assert_close(model(ids_cpu), expected_logits[name], rtol=1e-4, atol=1e-5)
    results["checkpoint_reloaded_and_checked"] = True
    (output_dir / "metrics.json").write_text(
        json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8"
    )

OUTPUT_DIR = Path.cwd() / "lesson_16_outputs"
save_experiment(models, results, OUTPUT_DIR)
print("保存并重新加载核对成功：", OUTPUT_DIR.resolve())

保存并重新加载核对成功： /mnt/data/_build16/notebook_run/lesson_16_outputs


## 14. 工程边界：为什么这仍不是长上下文部署实现？

本课显式创建 `[B,H,T,T]` 的分数和权重，单个矩阵元素数为 `B*H*T*T`，T 翻倍时元素数变为四倍。下面只计算理论存储大小，**不分配大矩阵，也不是 GPU 显存实测**。真正训练还包括其他激活、梯度、权重与优化器状态。

PyTorch 的 SDPA 可根据输入与后端选择不同实现；使用该接口不等于在任意机器上一定启用了某个 fused/Flash 内核，也不应把本课的速度当作推理框架基准。[3]

与第 14 课衔接时，必须另行设计 padding mask。`ignore_index=-100` 只忽略损失，不阻止 attention 读取 PAD。本课没有 PAD mask；将变长补齐数据直接送来，不能假装它自动正确屏蔽了 PAD。KV cache 中查询长度与键长度不同，也不在当前方形 mask 的适用范围内。

两层 Block 可以堆叠且保持因果性，但不应每层都重新添加一遍本课绝对位置 Embedding。下一课再把多层模型接回 next-token 目标和数据流程。

In [19]:
B, H, bytes_per_element = 2, 4, 4  # float32 单个元素 4 字节
for T in (1024, 2048, 4096):
    elements = B * H * T * T
    mib = elements * bytes_per_element / (1024 ** 2)
    print(f"T={T}: 单个 [B,H,T,T] float32 矩阵约 {mib:.0f} MiB")

T=1024: 单个 [B,H,T,T] float32 矩阵约 32 MiB
T=2048: 单个 [B,H,T,T] float32 矩阵约 128 MiB
T=4096: 单个 [B,H,T,T] float32 矩阵约 512 MiB


## 15. 独立练习：先补全，再打开检查开关

默认的 `RUN_EXERCISE_* = False` 只是让讲义可从头运行，不代表你完成了练习。补全代码后改为 True；检查器不依赖你手工抄写一个输出。

### 练习 A：拆头与合并

实现 `split_heads_ex(x, n_heads)` 和 `merge_heads_ex(heads)`，返回 `[B,H,T,d]` 和 `[B,T,D]`。不能直接调用正文同名功能函数。

约定输入是合法非空浮点张量；`split_heads_ex` 还需拒绝非正整数/布尔头数和 D 无法整除 H 的情况。检查会核对元素，而不只是核对形状；还会检查非连续输入与梯度。

In [20]:
def split_heads_ex(x, n_heads):
    raise NotImplementedError("完成 [B,T,D] -> [B,H,T,d]")


def merge_heads_ex(heads):
    raise NotImplementedError("完成逆变换，不能对头求平均")


def check_exercise_a(split_fn, merge_fn):
    x = torch.arange(60, dtype=torch.float64).reshape(2, 3, 10)
    got = split_fn(x, 2)
    torch.testing.assert_close(got[:, 0], x[:, :, :5])
    torch.testing.assert_close(got[:, 1], x[:, :, 5:])
    torch.testing.assert_close(merge_fn(got), x)
    noncontiguous = torch.randn(2, 10, 3, dtype=torch.float64).transpose(1, 2)
    torch.testing.assert_close(merge_fn(split_fn(noncontiguous, 2)), noncontiguous)
    xg = x.clone().requires_grad_()
    merge_fn(split_fn(xg, 2)).sum().backward()
    torch.testing.assert_close(xg.grad, torch.ones_like(xg))
    for bad_heads in (0, True, 3):
        try:
            split_fn(x, bad_heads)
        except ValueError:
            pass
        else:
            raise AssertionError("应拒绝非法头数")
    print("练习 A 检查通过")


RUN_EXERCISE_A = False
if RUN_EXERCISE_A:
    check_exercise_a(split_heads_ex, merge_heads_ex)
else:
    print("练习 A 尚未运行；补全后把开关设为 True")

练习 A 尚未运行；补全后把开关设为 True


### 练习 B：多头因果注意力核心

实现 `causal_attention_ex(q,k,v)`，返回 `(output, weights)`。输入已保证合法，为同形状 `[B,H,T,d]` 的 float64 张量，本练习无 Dropout。

必须自己完成点积、缩放、下三角 mask、softmax 和与 V 相乘；不能调用正文核心函数或官方 SDPA 来实现。检查器可以使用官方 SDPA 作为参考。

不要在 softmax **之后**才乘一个 0/1 mask；那会破坏归一化。注意缩放是每头维度 d。

In [21]:
def causal_attention_ex(q, k, v):
    raise NotImplementedError("实现多头因果注意力，返回 output 和 weights")


def check_exercise_b(fn):
    torch.manual_seed(169)
    arrays = [torch.randn(2, 2, 4, 3, dtype=torch.float64, requires_grad=True) for _ in range(3)]
    out, weights = fn(*arrays)
    reference = F.scaled_dot_product_attention(*arrays, is_causal=True, dropout_p=0.0)
    torch.testing.assert_close(out, reference)
    torch.testing.assert_close(weights.sum(-1), torch.ones(2, 2, 4, dtype=torch.float64))
    forbidden = torch.ones(4, 4, dtype=torch.bool).triu(1)
    if not bool((weights.masked_select(forbidden) == 0).all()):
        raise AssertionError("未来权重必须为零")
    upstream = torch.randn_like(out)
    grads_a = torch.autograd.grad((out * upstream).sum(), arrays)
    grads_b = torch.autograd.grad((reference * upstream).sum(), arrays)
    for ga, gb in zip(grads_a, grads_b):
        torch.testing.assert_close(ga, gb)
    q, k, v = [x.detach().clone() for x in arrays]
    base = fn(q, k, v)[0]
    k[:, :, 2:] += 8
    v[:, :, 2:] -= 9
    torch.testing.assert_close(fn(q, k, v)[0][:, :, :2], base[:, :, :2])
    print("练习 B 检查通过")


RUN_EXERCISE_B = False
if RUN_EXERCISE_B:
    check_exercise_b(causal_attention_ex)
else:
    print("练习 B 尚未运行；补全后把开关设为 True")

练习 B 尚未运行；补全后把开关设为 True


### 练习 C：写出一个完整 Block 的前向主干

实现：

```python
pre_norm_block_ex(x, norm1, attention, norm2, fc1, fc2)
```

传入的层都已正确构造，`attention(z)` 返回 `(attn_output, weights)`。本练习没有 Dropout。返回最终 `[B,T,D]` 隐藏状态。

要求保留两个残差主路，使用 GELU，第二个 LayerNorm 读取注意力残差相加后的状态；不要 `.detach()`。这相当于独立补写 `TransformerBlock.forward()` 的核心计算，再把它放进自己的类即可。

检查会比较输出、输入梯度，并归零两个分支检查恒等映射，避免“数值看似正确，但梯度主路断了”。

In [22]:
def pre_norm_block_ex(x, norm1, attention, norm2, fc1, fc2):
    raise NotImplementedError("完成两个 Pre-LN 残差子层")


def check_exercise_c(fn):
    torch.manual_seed(170)
    module = TransformerBlock(8, 2, 16).double().eval()
    xa = torch.randn(2, 3, 8, dtype=torch.float64, requires_grad=True)
    xb = xa.detach().clone().requires_grad_()
    out = fn(xa, module.norm1, module.attn, module.norm2, module.fc1, module.fc2)
    reference = module(xb)
    torch.testing.assert_close(out, reference)
    upstream = torch.randn_like(out)
    ga = torch.autograd.grad((out * upstream).sum(), xa)[0]
    gb = torch.autograd.grad((reference * upstream).sum(), xb)[0]
    torch.testing.assert_close(ga, gb)
    with torch.no_grad():
        module.attn.out_proj.weight.zero_()
        module.attn.out_proj.bias.zero_()
        module.fc2.weight.zero_()
        module.fc2.bias.zero_()
    xz = xa.detach().clone().requires_grad_()
    yz = fn(xz, module.norm1, module.attn, module.norm2, module.fc1, module.fc2)
    torch.testing.assert_close(yz, xz)
    yz.sum().backward()
    torch.testing.assert_close(xz.grad, torch.ones_like(xz))
    print("练习 C 检查通过")


RUN_EXERCISE_C = False
if RUN_EXERCISE_C:
    check_exercise_c(pre_norm_block_ex)
else:
    print("练习 C 尚未运行；补全后把开关设为 True")

练习 C 尚未运行；补全后把开关设为 True


---
## 16. 参考答案：先独立完成，再展开阅读

参考答案用不同名称，不覆盖练习函数。每个答案单元都包含自己的导入和最小自检，可以单独复制到空白 Notebook 运行；后面的综合核对使用前面的检查器。

### A：先拆特征，再交换轴；合并时反向操作

In [23]:
import torch


def split_heads_answer(x, n_heads):
    if type(n_heads) is not int or n_heads <= 0:
        raise ValueError("n_heads 必须为正整数")
    b, t, d_model = x.shape
    if d_model % n_heads:
        raise ValueError("D 必须能被 H 整除")
    return x.reshape(b, t, n_heads, d_model // n_heads).transpose(1, 2)


def merge_heads_answer(heads):
    b, h, t, d = heads.shape
    return heads.transpose(1, 2).contiguous().reshape(b, t, h * d)


answer_x = torch.arange(24, dtype=torch.float32).reshape(1, 3, 8)
torch.testing.assert_close(merge_heads_answer(split_heads_answer(answer_x, 2)), answer_x)
print("答案 A 最小自检通过")

答案 A 最小自检通过


### B：先屏蔽分数，再归一化

本答案按照练习的合法输入约定展示数学核心；正文实现另外含有输入检查。Q/K/V 都来自同一序列，因此这里的因果 mask 是方形。

In [24]:
import math
import torch


def causal_attention_answer(q, k, v):
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
    t = q.shape[-2]
    allowed = torch.ones(t, t, dtype=torch.bool, device=q.device).tril()
    weights = scores.masked_fill(~allowed, float("-inf")).softmax(dim=-1)
    return weights @ v, weights


answer_q = torch.zeros(1, 1, 2, 2, dtype=torch.float64)
answer_v = torch.tensor([[[[2., 0.], [4., 2.]]]], dtype=torch.float64)
answer_out, _ = causal_attention_answer(answer_q, answer_q, answer_v)
torch.testing.assert_close(answer_out, torch.tensor([[[[2., 0.], [3., 1.]]]], dtype=torch.float64))
print("答案 B 最小自检通过")

答案 B 最小自检通过


### C：第二条残差使用更新后的状态

最小自检中只用一个零注意力分支验证恒等路径；后面的综合核对会使用实际可学习注意力模块，并比较输入梯度。

In [25]:
import torch
from torch import nn
from torch.nn import functional as F


def pre_norm_block_answer(x, norm1, attention, norm2, fc1, fc2):
    attn_output, _ = attention(norm1(x))
    u = x + attn_output
    ffn_output = fc2(F.gelu(fc1(norm2(u))))
    return u + ffn_output


def zero_attention_for_answer(x):
    return torch.zeros_like(x), None


answer_norm1, answer_norm2 = nn.LayerNorm(4), nn.LayerNorm(4)
answer_fc1, answer_fc2 = nn.Linear(4, 8), nn.Linear(8, 4)
with torch.no_grad():
    answer_fc2.weight.zero_()
    answer_fc2.bias.zero_()
answer_hidden = torch.randn(1, 2, 4, requires_grad=True)
answer_result = pre_norm_block_answer(answer_hidden, answer_norm1, zero_attention_for_answer,
                                      answer_norm2, answer_fc1, answer_fc2)
torch.testing.assert_close(answer_result, answer_hidden)
answer_result.sum().backward()
torch.testing.assert_close(answer_hidden.grad, torch.ones_like(answer_hidden))
print("答案 C 最小自检通过")

答案 C 最小自检通过


### 用相同检查器核对参考答案

参考答案通过不等于你的练习通过。回到练习区，打开开关检查你自己的函数。

In [26]:
check_exercise_a(split_heads_answer, merge_heads_answer)
check_exercise_b(causal_attention_answer)
check_exercise_c(pre_norm_block_answer)

练习 A 检查通过
练习 B 检查通过
练习 C 检查通过


## 17. 验收与常见错误

完成后，应当能不看答案解释：为什么每个头仍处理所有位置；为什么缩放用每头维度；为什么允许对角线；为什么两个残差子层要顺序更新；为什么需要同时验证“能读前文”和“不读未来”。

| 现象 | 优先检查 |
|---|---|
| 形状正确但与官方输出不一致 | 拆头是否少了 transpose；合并是否先转回 `[B,T,H,d]` |
| 未来权重不为零 | mask 方向、True 语义、是否在 softmax 前处理 |
| 权重沿错误方向归一化 | 是否对 key 轴 `dim=-1` 做 softmax |
| 单头正常、多头不符 | 是否误用 `sqrt(D)` 代替 `sqrt(d)` |
| 因果性检查每次随机变化 | 是否仍处于 train 模式且启用了 Dropout |
| Block 归零分支后仍改变输入 | 是否改写了原始残差主路，或忘记归零 bias |
| 前向相同、梯度不同 | 是否出现 detach、重新构造张量、错误残差路径 |
| 最后 token 相同就输出相同 | 模型是否缺少有效的跨位置通道；是否仍是逐位置 MLP |

**下一课：第 17 课——从零组装微型 GPT 并进行字符级预训练。**

把多层因果 Block 接回 token/位置 Embedding、最终 LayerNorm、词表输出头、标签移位、批量训练与验证流程，再完成自回归生成。学会区分一个结构正确的微型语言模型与具备实际任务能力的预训练大模型。

## 18. 原始资料

本课数值例子、两样本探针、练习和测试均为自行构造的教学实验。下面的资料用于核对架构与 API，不作为本课实验结果的外部背书。在线文档可能重定向到更新版本，实际运行版本见开头输出。

[1] Vaswani et al., *Attention Is All You Need*, §3.1–3.3。
<https://arxiv.org/html/1706.03762v7>

[2] Xiong et al., *On Layer Normalization in the Transformer Architecture*。
<https://arxiv.org/abs/2002.04745>

[3] PyTorch, `scaled_dot_product_attention`：缩放、布尔 mask、显式 Dropout、后端差异。
<https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html>

[4] PyTorch, `nn.MultiheadAttention`：完整投影层、mask 语义、每头权重。
<https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html>

[5] PyTorch, `Tensor.view`：步长条件、与 reshape 的区别。
<https://docs.pytorch.org/docs/stable/generated/torch.Tensor.view.html>

[6] PyTorch, `nn.Dropout`：训练与评估行为。
<https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html>

[7] PyTorch 官方安装选择器。
<https://pytorch.org/get-started/locally/>

[8] PyTorch, `nn.Linear`：最后一维变换与权重存储形状。
<https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html>